# 🎙️ Agente Estudos — áudio do podcast (Kaggle / Colab, GPU)

Transforma os roteiros de `<pasta>/podcast/` em MP3, com uma voz para o **Professor** e outra para o **Aprendiz**. É o mesmo código do `python main.py --mode podcast-audio`, rodando com os motores que precisam de GPU.

**Antes de rodar**
- **Kaggle:** *Settings → Accelerator → GPU T4* e *Internet → On*. Suba o `podcast.zip` como Dataset (*Add Input*).
- **Colab:** *Ambiente de execução → Alterar tipo → GPU T4*. O upload do zip é feito na célula 4.
- No seu PC, zipe a pasta `podcast/` da pasta de estudos (só os roteiros bastam: `episodio_*.md`).
- O notebook clona o repositório do GitHub, então precisa de push feito. Se o repo for privado, crie o segredo `GITHUB_TOKEN`.

**Motores** (um por sessão: as dependências de um podem quebrar as do outro)

| Motor | Vozes | Clona voz | Licença do modelo |
|---|---|---|---|
| `xtts` | vozes prontas (liste na célula 5) | sim (`ref_wav`) | Coqui Public Model License, não comercial |
| `chatterbox` | **uma** voz padrão (sem clone, os dois papéis soam iguais) | sim (`ref_wav`) | MIT |
| `qwen3` | 9 vozes prontas, **nenhuma nativa em português** | sim (`ref_wav` + `ref_text`) | Apache-2.0 |
| `edge` | pt-BR nativas (Antonio, Francisca, Thalita) | não | serviço da Microsoft, sem GPU (melhor rodar no PC) |

**Fluxo sugerido:** `MODO = "voice-test"` em cada motor (o mesmo trecho de 5 falas, um MP3 por motor), escolher de ouvido e depois `MODO = "podcast-audio"` com o motor escolhido.

In [ ]:
# ── 1. Configuração (edite aqui) ──────────────────────────────────────────────
MODO   = "voice-test"        # "voice-test" (trecho fixo) | "podcast-audio" (episódios do zip)
ENGINE = "xtts"              # "xtts" | "chatterbox" | "qwen3" | "edge" — UM por sessão
EPISODIOS = []               # podcast-audio: [] = todos; ex.: [1, 2]

# Vozes fixas por papel (veja os nomes disponíveis na célula 5)
VOZES = {
    "xtts":   {"professor": "Damien Black", "aprendiz": "Ana Florence"},   # troque após listar tts.speakers
    "qwen3":  {"professor": "Ryan", "aprendiz": "Serena", "modelo": "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice"},
    "edge":   {"professor": "pt-BR-AntonioNeural", "aprendiz": "pt-BR-FranciscaNeural"},
    "chatterbox": {},
}

# Clonagem (opcional): caminho de um WAV de 10–20 s com a voz desejada.
# Kaggle: suba junto no Dataset; Colab: envie na célula 4. qwen3 exige também a transcrição (ref_text).
CLONE = {
    "professor": {"ref_wav": None, "ref_text": None},
    "aprendiz":  {"ref_wav": None, "ref_text": None},
}

REPO_URL = "https://github.com/Roberto207/agente_estudos"
BRANCH   = "main"

In [ ]:
# ── 2. Ambiente ───────────────────────────────────────────────────────────────
import os, re, subprocess, sys, shutil, zipfile
from pathlib import Path

NO_KAGGLE = Path("/kaggle/working").exists()
NO_COLAB = "google.colab" in sys.modules
WORK = Path("/kaggle/working") if NO_KAGGLE else Path("/content") if NO_COLAB else Path.cwd()
# No Kaggle, tudo em /kaggle/working vira "Output" da versão: o repositório e o cache de
# áudio (~50 MB por episódio) vão para /kaggle/temp, e só os zips ficam no Output.
TEMP = Path("/kaggle/temp") if NO_KAGGLE else WORK
TEMP.mkdir(parents=True, exist_ok=True)
print("Ambiente:", "Kaggle" if NO_KAGGLE else "Colab" if NO_COLAB else "local", "| saída em", WORK, "| temporários em", TEMP)
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip() if shutil.which("nvidia-smi") else ""
print("GPU:", gpu or "nenhuma — xtts/chatterbox ficam muito lentos e qwen3 não roda")
print("ffmpeg:", shutil.which("ffmpeg") or "AUSENTE (instale: apt-get install -y ffmpeg)")

def segredo(nome: str) -> str:
    if NO_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(nome)
        except Exception:
            pass
    if NO_COLAB:
        try:
            from google.colab import userdata
            return userdata.get(nome)
        except Exception:
            pass
    return os.environ.get(nome, "")

In [ ]:
# ── 3. Código do projeto + motor escolhido ────────────────────────────────────
REPO_DIR = TEMP / "agente_estudos"
if not REPO_DIR.exists():
    token = segredo("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO_DIR)], capture_output=True, text=True)
    if r.returncode != 0:  # não imprime a URL (pode conter o token)
        raise RuntimeError("git clone falhou — repo privado sem GITHUB_TOKEN? " + r.stderr.replace(token or "\0", "***")[-500:])
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Núcleo (roteiro, normalização, montagem) + só o motor desta sessão.
PACOTES = {
    # XTTS não importa com transformers 5.x (idiap/coqui-ai-TTS#558); [codec] é exigido com torch >= 2.9.
    "xtts": ["coqui-tts[codec]", "transformers>=4.43,<5"],
    "chatterbox": ["chatterbox-tts"],        # pode trocar a versão do torch: se der erro, reinicie a sessão
    "qwen3": ["qwen-tts", "soundfile"],
    "edge": ["edge-tts"],
}
IMPORTA = {
    "xtts": "from TTS.api import TTS",
    "chatterbox": "from chatterbox.mtl_tts import ChatterboxMultilingualTTS",
    "qwen3": "from qwen_tts import Qwen3TTSModel",
    "edge": "import edge_tts",
}
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "num2words", "pyyaml", "rich", *PACOTES[ENGINE]], check=True)
# Testa o import num processo novo: falha AQUI, com o erro real, e não na célula 6.
r = subprocess.run([sys.executable, "-c", IMPORTA[ENGINE]], capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError(f"{ENGINE}: o pacote instalou mas não importa — veja o erro acima. "
                       "Se mudou a versão do torch/transformers, reinicie a sessão e rode de novo.")
print("instalado e importável:", ", ".join(PACOTES[ENGINE]))

In [ ]:
# ── 4. Entrada: roteiros (podcast-audio) e áudios de referência (clone) ──────
PASTA = WORK / "estudo"
PODCAST = PASTA / "podcast"

def _extrair(zip_path: Path):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(PASTA / "_zip")
    # aceita zip com a pasta "podcast/" dentro ou com os episodio_*.md soltos
    achados = sorted((PASTA / "_zip").rglob("episodio_*.md"))
    if not achados:
        raise RuntimeError(f"nenhum episodio_*.md dentro de {zip_path.name}")
    PODCAST.mkdir(parents=True, exist_ok=True)
    for f in achados:
        shutil.copy(f, PODCAST / f.name)
    for extra in (PASTA / "_zip").rglob("serie.json"):
        shutil.copy(extra, PODCAST / "serie.json")
    print(f"{len(achados)} roteiro(s) em {PODCAST}")

if MODO == "podcast-audio" and not any(PODCAST.glob("episodio_*.md")):
    if NO_KAGGLE:
        zips = sorted(Path("/kaggle/input").rglob("*.zip"))
        soltos = sorted(Path("/kaggle/input").rglob("episodio_*.md"))
        if zips:
            _extrair(zips[0])
        elif soltos:  # o Kaggle às vezes descompacta o dataset sozinho
            PODCAST.mkdir(parents=True, exist_ok=True)
            for f in soltos:
                shutil.copy(f, PODCAST / f.name)
            print(f"{len(soltos)} roteiro(s) em {PODCAST}")
        else:
            raise RuntimeError("suba o podcast.zip como Dataset (Add Input)")
    elif NO_COLAB:
        from google.colab import files
        print("Envie o podcast.zip (e, se for clonar, os WAVs de referência):")
        for nome in files.upload():
            if nome.endswith(".zip"):
                _extrair(Path(nome))
    else:
        raise RuntimeError(f"coloque os roteiros em {PODCAST}")

for papel, c in CLONE.items():  # resolve caminhos relativos dos WAVs de referência
    if c["ref_wav"] and not Path(c["ref_wav"]).is_absolute():
        candidatos = list(WORK.rglob(Path(c["ref_wav"]).name)) + list(Path("/kaggle/input").rglob(Path(c["ref_wav"]).name) if NO_KAGGLE else [])
        if candidatos:
            c["ref_wav"] = str(candidatos[0])
    if c["ref_wav"]:
        print(f"clone {papel}: {c['ref_wav']}")

In [ ]:
# ── 5. (Opcional) Listar vozes do motor — rode antes de escolher VOZES ────────
from src.tts.base import config_voz
from src.tts import registry
cfg_teste = config_voz({"voz": {"engines": {ENGINE: VOZES.get(ENGINE, {})}}})
try:  # opcional: um erro aqui não deve derrubar o "Save & Run All" do Kaggle
    eng = registry.get_engine(ENGINE, cfg_teste)
    if ENGINE == "xtts":
        print("Vozes XTTS:", eng.vozes())
    elif ENGINE == "qwen3":
        m = eng._carregar(eng.id_modelo)
        print("Vozes:", m.get_supported_speakers())
        print("Idiomas:", m.get_supported_languages())
    elif ENGINE == "chatterbox":
        print("Aceita áudio de referência:", eng.aceita_referencia())
    else:
        print("edge: pt-BR-AntonioNeural, pt-BR-FranciscaNeural, pt-BR-ThalitaMultilingualNeural")
except Exception as exc:
    print(f"(não consegui listar as vozes: {exc})")

In [ ]:
# ── 6. Gerar ──────────────────────────────────────────────────────────────────
import time
config = {"voz": {
    "engine": ENGINE,
    "engines": {ENGINE: VOZES.get(ENGINE, {})},
    "clone": CLONE,
    "cache_dir": str(TEMP / "cache_tts"),   # cache por fala: reexecutar na mesma sessão só refaz o que falta
    "teste": {"engines": [ENGINE]},
}}
t0 = time.time()
if MODO == "voice-test":
    from src.tts.voice_test import run as voice_test
    SAIDA = WORK / "teste_vozes"
    rel = voice_test(config, saida=str(SAIDA), engines=[ENGINE])
    for r in rel["resultados"]:
        print(r["engine"], "OK" if r["ok"] else f"FALHOU: {r['erro']}")
else:
    from src.tts.audio import run as audio
    SAIDA = PODCAST / "audio"   # PODCAST fica em WORK: no Kaggle, os MP3 também aparecem no Output
    audio(config, str(PASTA), episodios=EPISODIOS or None, engine_nome=ENGINE)
print(f"\nConcluído em {(time.time() - t0) / 60:.1f} min")

In [ ]:
# ── 7. Ouvir aqui e baixar ────────────────────────────────────────────────────
from IPython.display import Audio, display
mp3s = sorted(SAIDA.glob("*.mp3"))
for f in mp3s[:3]:
    print(f.name); display(Audio(str(f)))
zip_path = shutil.make_archive(str(WORK / f"audio_{ENGINE}_{MODO}"), "zip", SAIDA)
print("Zip:", zip_path)
if NO_COLAB:
    from google.colab import files
    files.download(zip_path)
elif NO_KAGGLE:
    print("Baixe pela aba Output do notebook (lado direito) → arquivo .zip")

### Depois de baixar
- **voice-test:** ouça `<motor>.mp3` de cada motor e anote o melhor. Fixe-o em `voz.engine` no `config.yaml`, ou use-o neste notebook com `MODO = "podcast-audio"`.
- **podcast-audio:** descompacte os MP3 em `<pasta>/podcast/audio/` no vault. Cada MP3 vem com um `.json` ao lado, com motor, vozes, duração e o hash do roteiro de origem.